In [ ]:

import os
import pandas as pd
import requests
from dotenv import load_dotenv
from time import sleep

# === CONFIG ===
INPUT_FILE = r"C:\GitHub\Android-Mobile-Apps\Repo_List_checked_Standard_Android_app.xlsx"
OUTPUT_FILE = r"C:\GitHub\Android-Mobile-Apps\Repo_List_enriched.xlsx"
REQUEST_DELAY = 1  # seconds between GitHub API requests

# === LOAD GITHUB TOKEN ===

load_dotenv("All_Token.env")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    raise ValueError("GITHUB_TOKEN not found in All_Token.env")

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "User-Agent": "android-repo-analyzer"
}

# === LOAD DATA ===
df = pd.read_excel(INPUT_FILE)
df = df[(df["has_manifest"] == "yes") & (df["has_activity"] == "yes")].copy()

# === HELPER FUNCTION ===
def fetch_github_metadata(full_name):
    url = f"https://api.github.com/repos/{full_name}"
    try:
        r = requests.get(url, headers=HEADERS)
        if r.status_code != 200:
            return None
        data = r.json()
        readme_url = f"https://api.github.com/repos/{full_name}/readme"
        readme_resp = requests.get(readme_url, headers=HEADERS)
        readme_text = ""
        if readme_resp.status_code == 200:
            readme_text = readme_resp.json().get("content", "")
        return {
            "description": data.get("description", ""),
            "topics": ", ".join(data.get("topics", [])),
            "language": data.get("language", ""),
            "readme": readme_text
        }
    except Exception as e:
        print(f"Error fetching {full_name}: {e}")
        return None

def flag_library_or_demo(name, description, readme):
    keywords = ["library", "demo", "sample", "example", "template", "plugin", "test", "benchmark"]
    text = f"{name} {description} {readme}".lower()
    return any(k in text for k in keywords)

# === ENRICH DATA ===
enriched_data = []
for idx, row in df.iterrows():
    full_name = row["full_name"]
    metadata = fetch_github_metadata(full_name)
    if metadata:
        flagged = flag_library_or_demo(full_name, metadata["description"], metadata["readme"])
        enriched_data.append({
            "full_name": full_name,
            "description": metadata["description"],
            "topics": metadata["topics"],
            "language": metadata["language"],
            "is_library_or_demo": flagged
        })
    else:
        enriched_data.append({
            "full_name": full_name,
            "description": "",
            "topics": "",
            "language": "",
            "is_library_or_demo": False
        })
    print(f"Processed: {full_name}")
    sleep(REQUEST_DELAY)

# === MERGE AND SAVE ===
df_enriched = pd.DataFrame(enriched_data)
df_final = df.merge(df_enriched, on="full_name", how="left")
df_final.to_excel(OUTPUT_FILE, index=False)
print(f"✅ Enriched data saved to: {OUTPUT_FILE}")


Processed: 008chen/InterpolatorShow
Processed: 00ec454/pop
Processed: 06peng/FrescoDemo
Processed: 0ranko0P/AutoDark
Processed: 0x4f53/Wristkey
Processed: 0xbad1d3a5/Kaku
Processed: 0xf104a/NextcloudServices
Processed: 0xm1nam0/RxCore
Processed: 0xZhangKe/Fread
Processed: 0xZhangKe/ShiZhong
Processed: 100mslive/100ms-android
Processed: 109021017/android-TopActivity
Processed: 10clouds/FluidBottomNavigation-android
Processed: 10miaomiao/bilimiao2
Processed: 1170762202/WanAndroid
Processed: 123lxw123/VideoWorld_Android
Processed: 123ufo/DWRefreshLayout
Processed: 1250422131/bilibilias
Processed: 13767004362/Camera2App
Processed: 13767004362/HookDemo
Processed: 15527621771/Android-Advert-SDK
Processed: 1596941391qq/pokerogue-android
Processed: 15dd/wenku8reader
Processed: 18601949127/DiDiCallCar
Processed: 18702953620/DouDemo
Processed: 18702953620/MVPDemo
Processed: 1900Star/MusicPlayer-Smartisan
Processed: 1993hzw/Doodle
Processed: 1993hzw/Graffiti
Processed: 1993hzw/TiledMapView
Proces